# 05 — Things single-cell RNA-seq cannot do

**Day 2, 11:30–12:15**

This is the notebook the whole workshop is built around. Four analyses, each of
which has **no possible scRNA-seq equivalent** — not "harder", not "less powerful".
Impossible, because the measurement was thrown away at dissociation.

1. **Distance to an anatomical structure** as a continuous covariate
2. **Direct cell–cell contact** with a spatially correct null
3. **Sub-cellular transcript localisation** — nuclear vs cytoplasmic, per gene
4. **Segmentation-free signal** — what is there before anyone drew a polygon

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc
import scipy.sparse as sp
import seaborn as sns
from scipy.spatial import cKDTree

sc.settings.verbosity = 1
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA = ROOT / "data"
NAVY, GOLD, CORAL, ICE = "#001158", "#FBAE40", "#F26B43", "#BCD2FF"

adata = sc.read_h5ad(DATA / "ovarian_niches.h5ad")   # from notebook 04
x, y = adata.obsm["spatial"].T
types = list(adata.obs["cell_type"].cat.categories)
types

## 1. Distance to the tumour boundary

In scRNA-seq, "distance from the tumour edge" does not exist as a variable. Here it
is just geometry, and it turns a categorical comparison into a **continuous dose–response**.

The recipe: pick a reference population, build a KD-tree of its coordinates, and ask
every other cell how far away it is.

In [ ]:
def biggest_matching(*patterns, default_idx=0):
    """The most abundant cell type whose name contains any of these strings.

    Picking the *largest* match matters now that the annotation has several
    tumour and several fibroblast subtypes: an alphabetical first-match would
    happily choose a rare proliferating subcluster as the tumour reference.
    """
    counts = adata.obs["cell_type"].value_counts()
    hits = [t for t in counts.index
            if any(p.lower() in str(t).lower() for p in patterns)]
    return hits[0] if hits else types[default_idx]


TUMOUR = biggest_matching("tumour", "epithel")
print("reference population:", TUMOUR)

is_tum = (adata.obs["cell_type"] == TUMOUR).to_numpy()
tree = cKDTree(adata.obsm["spatial"][is_tum])
dist, _ = tree.query(adata.obsm["spatial"], k=1)
adata.obs["dist_to_tumour"] = dist
adata.obs.loc[is_tum, "dist_to_tumour"] = 0.0

print(adata.obs.loc[~is_tum, "dist_to_tumour"].describe().round(1))

> **A caution before the pretty plot.** Nearest-neighbour distance to a *cell type*
> is not the same as distance to a *structure*. A single misclassified tumour cell
> sitting alone in the stroma creates a false "tumour" 200 um from any nest, and
> every cell around it gets a distance of ~10 um. Two defences: require a minimum
> local density of tumour cells before a point counts as tumour, or build the
> reference from your **niche** labels (notebook 04) instead of cell types, since
> niches are already smoothed over neighbourhoods.

In [ ]:
# more robust reference: tumour cells that are themselves in a tumour-dominated niche
tum_niche = (
    adata.obs.groupby("niche", observed=True)
    .apply(lambda d: (d["cell_type"] == TUMOUR).mean())
    .idxmax()
)
core = is_tum & (adata.obs["niche"] == tum_niche).to_numpy()
print(f"tumour-dominated niche: {tum_niche}  ({core.sum():,} reference cells)")

tree = cKDTree(adata.obsm["spatial"][core])
d2, _ = tree.query(adata.obsm["spatial"], k=1)
adata.obs["dist_to_nest"] = d2

fig, axes = plt.subplots(1, 2, figsize=(13, 5.6))
pc = axes[0].scatter(x, y, c=np.clip(d2, 0, 300), s=1.1, cmap="viridis",
                     linewidths=0, rasterized=True)
plt.colorbar(pc, ax=axes[0], label="distance to tumour nest (um)")
axes[0].set_title("distance field")
axes[1].scatter(x, y, s=0.6, c="0.88", linewidths=0, rasterized=True)
axes[1].scatter(x[core], y[core], s=1.6, c=NAVY, linewidths=0, rasterized=True)
axes[1].set_title("reference population")
for ax in axes:
    ax.set_aspect("equal"); ax.invert_yaxis(); ax.set_xticks([]); ax.set_yticks([])
plt.tight_layout(); plt.show()

In [ ]:
# Composition as a function of distance — the invasive front, quantified
bins = [0, 10, 25, 50, 100, 200, 400, np.inf]
labels = ["0-10", "10-25", "25-50", "50-100", "100-200", "200-400", ">400"]
adata.obs["dist_bin"] = pd.cut(adata.obs["dist_to_nest"], bins=bins, labels=labels)

frac = (
    pd.crosstab(adata.obs["dist_bin"], adata.obs["cell_type"], normalize="index")
    .drop(columns=[TUMOUR], errors="ignore")
)
fig, ax = plt.subplots(figsize=(8, 4.6))
frac.plot(kind="bar", stacked=True, ax=ax, colormap="tab20", width=0.85)
ax.set_xlabel("distance from tumour nest (um)"); ax.set_ylabel("fraction of cells")
ax.legend(loc="center left", bbox_to_anchor=(1.01, 0.5), frameon=False, fontsize=8)
ax.set_title("tissue composition as a function of distance")
plt.tight_layout(); plt.show()

> **Try it yourself — how far does the effect reach?**
>
> Pick a cell type and see how its abundance changes with distance from the tumour.
> One number per distance band, so you can read the trend without interpreting a
> stacked bar chart.

In [ ]:
CELL_TYPE = adata.obs["cell_type"].value_counts().index[1]    # <-- CHANGE THIS

print(f"cell types available:\n{list(adata.obs['cell_type'].cat.categories)}\n")
print(f"showing: {CELL_TYPE}\n")

tab = pd.crosstab(adata.obs["dist_bin"], adata.obs["cell_type"], normalize="index")
if CELL_TYPE not in tab.columns:
    raise KeyError(f"{CELL_TYPE!r} is not one of the cell types listed above")
frac_by_band = tab[CELL_TYPE]

for band, frac in frac_by_band.items():
    bar = "#" * int(round(100 * frac / max(frac_by_band.max(), 1e-9) * 0.4))
    print(f"  {str(band):>9} um   {100 * frac:5.1f}%  {bar}")

In [ ]:
# Gene expression as a function of distance, within ONE cell type.
# This is the design that has no scRNA-seq equivalent: same cell type,
# different position, so position is the only variable.
CELLTYPE = biggest_matching("fibro", "caf", default_idx=min(1, len(types) - 1))
sub = adata[(adata.obs["cell_type"] == CELLTYPE) & (adata.obs["dist_to_nest"] < 400)].copy()
print(f"{CELLTYPE}: {sub.n_obs:,} cells within 400 um of a nest")

# genes that actually vary among the fibroblasts in this section
genes = [g for g in ["POSTN", "COL1A1", "DCN", "LUM", "TIMP3", "MFAP5", "C7"]
         if g in sub.var_names][:4]
X = sub[:, genes].X
X = X.toarray() if hasattr(X, "toarray") else np.asarray(X)

fig, axes = plt.subplots(1, len(genes), figsize=(4.0 * len(genes), 3.6), sharex=True)
axes = np.atleast_1d(axes)
edges = np.arange(0, 401, 25)
centres = 0.5 * (edges[:-1] + edges[1:])
which = np.digitize(sub.obs["dist_to_nest"], edges) - 1
for k, (g, ax) in enumerate(zip(genes, axes)):
    means = [X[which == i, k].mean() if (which == i).sum() > 20 else np.nan
             for i in range(len(centres))]
    sems = [X[which == i, k].std() / max(np.sqrt((which == i).sum()), 1)
            if (which == i).sum() > 20 else np.nan for i in range(len(centres))]
    means, sems = np.array(means), np.array(sems)
    ax.plot(centres, means, color=NAVY, lw=2)
    ax.fill_between(centres, means - 1.96 * sems, means + 1.96 * sems, color=ICE, alpha=0.6)
    ax.set_title(g); ax.set_xlabel("distance to nest (um)")
axes[0].set_ylabel(f"mean expression\n({CELLTYPE})")
sns.despine(); plt.tight_layout(); plt.show()

A sloping line here is a **spatial gradient of cell state within one cell type**.
If you had dissociated this tissue, every one of those cells would have collapsed
into a single point in the fibroblast cluster and the gradient would be invisible —
you might have called it "CAF heterogeneity" and gone looking for subclusters.

### Exercise 5.1
Do the same for an immune population, and for distance to **vessels** instead of
tumour (use the endothelial cells as reference). Does any gene respond to one
distance and not the other? Watch out: the two distances are correlated, so
condition on both before claiming either.

In [ ]:
# your code here

## 2. Cell–cell contact, with a null that respects geometry

"Cell type A talks to cell type B" is inferred in scRNA-seq from ligand and receptor
expression in two clusters — with no evidence the cells were ever within a millimetre
of each other. Here you can require **actual contact**.

The null model is the delicate part. Shuffling labels at random destroys tissue
structure and makes everything look significant. A better null keeps each cell's
neighbourhood and shuffles labels **within distance-matched strata**, or permutes
labels only among cells of similar local density. Below we use the simplest defensible
version: permute labels within niche, so the null preserves large-scale architecture
and only tests fine-scale arrangement.

In [ ]:
A = adata.obsp["spatial_connectivities"].tocsr()
labels = adata.obs["cell_type"].to_numpy()
niches = adata.obs["niche"].to_numpy()

def contact_counts(lab):
    oh = pd.get_dummies(pd.Categorical(lab, categories=types)).to_numpy().astype(float)
    return oh.T @ (A @ oh)

obs_counts = contact_counts(labels)

rng = np.random.default_rng(0)
N_PERM = 200
null = np.zeros((N_PERM, len(types), len(types)))
for p in range(N_PERM):
    perm = labels.copy()
    for n in np.unique(niches):                    # shuffle WITHIN niche
        m = niches == n
        perm[m] = rng.permutation(perm[m])
    null[p] = contact_counts(perm)

z = (obs_counts - null.mean(0)) / (null.std(0) + 1e-9)
z_df = pd.DataFrame(z, index=types, columns=types)

fig, ax = plt.subplots(figsize=(7.5, 6))
sns.heatmap(z_df, cmap="RdBu_r", center=0, annot=True, fmt=".0f", ax=ax,
            cbar_kws={"label": "z vs within-niche null"})
ax.set_title("contact enrichment, architecture-preserving null")
plt.tight_layout(); plt.show()

Compare this with `sq.gr.nhood_enrichment` from notebook 04. **The z-scores are much
smaller.** The global-shuffle null was crediting the method for structure you could
already see with your eyes; this one only reports arrangement beyond the niche level.

Whenever a spatial result looks spectacular, ask what the null was. Most of the
time, the null was too easy to beat.

In [ ]:
# Ligand-receptor, spatially constrained: does a specific L-R pair sit across contacts
# more than expected? Self-contained; no database download needed.
PAIR = ("CXCL12", "CXCR4")
if all(g in adata.var_names for g in PAIR):
    L = np.asarray(adata[:, PAIR[0]].X.todense()).ravel()
    R = np.asarray(adata[:, PAIR[1]].X.todense()).ravel()
    lpos, rpos = L > np.percentile(L, 90), R > np.percentile(R, 90)

    obs_edges = float(lpos @ (A @ rpos))
    perm_edges = []
    for _ in range(500):
        pr = rng.permutation(rpos)
        perm_edges.append(float(lpos @ (A @ pr)))
    perm_edges = np.array(perm_edges)
    zz = (obs_edges - perm_edges.mean()) / perm_edges.std()
    print(f"{PAIR[0]}-high  ->  {PAIR[1]}-high contacts")
    print(f"  observed {obs_edges:,.0f}   null {perm_edges.mean():,.0f} +/- {perm_edges.std():,.0f}")
    print(f"  z = {zz:.1f}")
else:
    print(f"{PAIR} not both on the panel — pick another pair present in adata.var_names")

> **Optional:** `sq.gr.ligrec()` runs this systematically against the OmniPath
> database, but it downloads the database on first use. If the workshop wifi
> cooperates, try it; if not, the hand-rolled version above is the same idea and you
> can see every moving part.

## 3. Sub-cellular localisation

Now we go below the cell. The transcript table records, per molecule, whether it fell
inside the nucleus. Aggregate that per gene and you have measured **nuclear versus
cytoplasmic localisation, in situ, for 5,000 genes at once**.

There is no version of this experiment in scRNA-seq. Nuclear and cytoplasmic
transcripts go into the same droplet and the same count.

In [ ]:
tx = pd.read_parquet(DATA / "transcripts_crop.parquet")
tx = tx[tx["qv"] >= 20]
print(f"{len(tx):,} high-confidence transcripts")

if "overlaps_nucleus" in tx.columns:
    per_gene = (
        tx.groupby("feature_name")
        .agg(n=("overlaps_nucleus", "size"), nuclear=("overlaps_nucleus", "mean"))
        .query("n >= 200")
        .sort_values("nuclear")
    )
    print("\nmost cytoplasmic genes:"); display(per_gene.head(10).round(3))
    print("most nuclear genes:"); display(per_gene.tail(10).round(3))
else:
    print("no overlaps_nucleus column; compute it from the nucleus polygons instead")

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(per_gene["nuclear"], bins=50, color=NAVY)
ax.axvline(per_gene["nuclear"].median(), color=GOLD, lw=2, label="median gene")
ax.set_xlabel("fraction of transcripts overlapping the nucleus")
ax.set_ylabel("genes"); ax.legend(); sns.despine(); plt.show()

The genes at the top of that list are usually the ones you would predict —
nuclear-retained lncRNAs (`MALAT1`, `NEAT1`), and genes with slow export. The genes
at the bottom are typically abundant secretory and structural mRNAs translated on the
rough ER.

**Why an anatomy department should care:** this is a proxy for transcription *timing*.
A nuclear-skewed profile for an inducible gene means recent transcription; a
cytoplasmic-skewed profile means the message has been out for a while. In a
development or regeneration context, that is a read on which cells switched on
recently — a temporal axis, from a fixed section.

Two honest caveats: the nucleus polygon has its own segmentation error, and a 2D
section through a 3D nucleus mislabels transcripts near the top and bottom.
Treat this as comparative across genes, not as an absolute measurement.

In [ ]:
# Look at it directly: one nuclear-skewed and one cytoplasmic-skewed gene,
# plotted over the nucleus outlines.
nucs = pd.read_parquet(DATA / "nucleus_boundaries_crop.parquet")
g_nuc = per_gene.index[-1]
g_cyt = per_gene.query("n >= 1000").index[0]

x0, y0, W = tx["x_location"].min() + 120, tx["y_location"].min() + 120, 90
fig, axes = plt.subplots(1, 2, figsize=(12, 6))
for ax, g, col in [(axes[0], g_nuc, NAVY), (axes[1], g_cyt, CORAL)]:
    ids = nucs[(nucs["vertex_x"].between(x0, x0 + W)) & (nucs["vertex_y"].between(y0, y0 + W))]["cell_id"].unique()
    for cid in ids:
        p = nucs[nucs["cell_id"] == cid]
        ax.fill(p["vertex_x"], p["vertex_y"], color="0.88", lw=0.5, edgecolor="0.6")
    s = tx[(tx["feature_name"] == g) & tx["x_location"].between(x0, x0 + W)
           & tx["y_location"].between(y0, y0 + W)]
    ax.scatter(s["x_location"], s["y_location"], s=16, c=col, linewidths=0)
    ax.set_xlim(x0, x0 + W); ax.set_ylim(y0, y0 + W)
    ax.set_aspect("equal"); ax.invert_yaxis(); ax.set_xticks([]); ax.set_yticks([])
    ax.set_title(f"{g}  (nuclear fraction {per_gene.loc[g,'nuclear']:.2f})")
plt.tight_layout(); plt.show()

### Exercise 5.2
Does the nuclear fraction of a gene change between **niches**? Restrict to one gene
with plenty of counts, assign each transcript to a niche via its cell, and compare.
A gene that is more nuclear at the invasive front than in the nest core is a claim
about transcriptional dynamics in space.

In [ ]:
# your code here

## 4. The doublet you cannot see in 2D  *(optional)*

Section 3 used the z coordinate to ask *where in the cell* a transcript sits. Now use
it to ask a harder question: **is this even one cell?**

A Xenium section is 5–10 µm thick, and both the segmentation and everything we have
done so far treat it as flat. But two cells can sit on top of each other within that
thickness. Projected to 2D they become one polygon with one mixed profile — a
**vertical doublet**. It is not the segmentation's fault; the information was
destroyed by flattening.

This is a different failure from the 2D merge we met in notebook 02:

| | 2D merge | Vertical doublet |
|---|---|---|
| Cause | polygon boundary drawn wrong | two cells stacked in z |
| Visible in the DAPI image? | usually, as two nuclei | often not — one nucleus hides the other |
| Fixed by re-segmenting in 2D? | often | never |
| Detected by | looking at the polygon | comparing top and bottom of the slice |

**ovrlpy** ([Tiesmeyer et al., *Nature Biotechnology* 2026](https://doi.org/10.1038/s41587-026-03004-8))
splits the section into a virtual top and bottom half, embeds the transcriptome of
each, and asks how similar they are at every position. The result is a **vertical
signal integrity (VSI)** map: high where the two halves agree, low where they do not.
Low VSI means either two different cells stacked, or a tissue fold.

It is segmentation-free, so it is an *independent* check on everything downstream.

In [ ]:
# Optional section. Install with:  pip install ovrlpy
# Roughly 1-3 minutes on the teaching window; longer on a full section.
try:
    import ovrlpy

    HAVE_OVRLPY = True
    print("ovrlpy", getattr(ovrlpy, "__version__", "installed"))
except ImportError:
    HAVE_OVRLPY = False
    print("ovrlpy not installed — skipping section 4.")
    print("Install it with:  pip install ovrlpy")

In [ ]:
if HAVE_OVRLPY:
    # ovrlpy wants transcripts with x, y, z in microns plus a gene column. The io
    # helpers handle the platform-specific column names for you; we also ask for
    # cell_id so we can map the result back onto our cells later.
    transcripts = ovrlpy.io.read_Xenium(
        DATA / "transcripts_crop.parquet",
        additional_columns=["cell_id"],
    )
    print(transcripts.head())

    ovr = ovrlpy.Ovrlp(
        transcripts,
        KDE_bandwidth=2.5,   # smoothness, in um — roughly a cell radius
        n_components=20,     # PCA components for the local transcriptome embedding
        n_workers=4,
        random_state=0,
    )
    ovr.analyse(min_transcripts=20)
    print("done")

In [ ]:
if HAVE_OVRLPY:
    # The VSI map. Bright = top and bottom halves agree; dark = they do not.
    fig = ovrlpy.plot_signal_integrity(ovr, signal_threshold=4)
    plt.show()

### Reading the VSI map

`signal_threshold` hides pixels with too little signal to judge — a low VSI on three
transcripts means nothing. Raise it if the map looks like static.

What the dark regions mean:

- **Small dark spots, roughly cell-sized** — genuine vertical doublets, two cells
  stacked in the section.
- **Long dark streaks or arcs** — a tissue fold. Section damage, not biology. This is
  the same thing you were told to look for in notebook 02, but VSI finds folds that
  are invisible in a count-density map.
- **Dark at the tissue edge** — usually thickness falling off; not informative.

The paper suggests **VSI < 0.7** as a working threshold for flagging doublets, with
the caveat that it depends on panel and tissue. Treat it as a starting point you
check, not a constant.

In [ ]:
if HAVE_OVRLPY:
    doublets = ovr.detect_doublets(min_signal=4, integrity_sigma=1)
    print(f"{len(doublets)} candidate overlap events")
    print(doublets.head())

In [ ]:
if HAVE_OVRLPY and len(doublets):
    # Look at one. The panel shows top half, bottom half, and a side view, so you
    # can see the two transcriptomes separating in z.
    xd, yd = doublets["x", "y"].row(0)
    fig = ovrlpy.plot_region_of_interest(ovr, xd, yd, window_size=60)
    plt.show()
    print(f"doublet at ({xd:.0f}, {yd:.0f}) um")

### Bringing VSI back to your cells

A map is interesting; a **per-cell QC column** is usable. `cell_integrity_from_transcripts`
gives VSI per pixel per cell, which we summarise two ways — mean VSI, and the fraction
of a cell's pixels below threshold — and attach to `adata.obs`.

Then the question that matters: **do the low-VSI cells sit anywhere in particular?**

The next three cells are deliberately split so you can see where things go wrong:
compute, then aggregate, then join. Each prints what it found. ovrlpy's output
schema and the way Xenium marks unassigned transcripts have both changed between
versions, so the code detects them rather than assuming — if a cell raises, the
printout above it tells you which name to adjust.


In [ ]:
if HAVE_OVRLPY:
    import polars as pl

    # What marks "no cell"? Xenium writes the string "UNASSIGNED"; some exports
    # use -1 or 0. Read it off the data rather than assuming.
    ids = transcripts["cell_id"]
    if ids.dtype == pl.Utf8:
        counts = ids.value_counts(sort=True).head(3)
        unassigned = "UNASSIGNED" if (ids == "UNASSIGNED").any() else counts["cell_id"][0]
    else:
        unassigned = -1 if (ids == -1).any() else 0
    print(f"cell_id dtype {ids.dtype}, treating {unassigned!r} as unassigned")

    cell_px = ovrlpy.cell_integrity_from_transcripts(
        ovr, cell_id="cell_id", unassigned=unassigned
    )
    print("returned columns:", cell_px.columns)
    print(cell_px.head())

In [ ]:
if HAVE_OVRLPY:
    # Column names have moved between ovrlpy versions, so find them rather than
    # hard-coding. If this raises, print cell_px.columns above and adjust.
    def pick(df, *candidates, required=True):
        for name in candidates:
            if name in df.columns:
                return name
        if required:
            raise KeyError(f"none of {candidates} in {df.columns}")
        return None

    VSI_COL = pick(cell_px, "vsi", "integrity", "signal_integrity", "VSI")
    ID_COL = pick(cell_px, "cell_id", "cell", "segment_id")
    SIG_COL = pick(cell_px, "signal", "signal_strength", required=False)
    print(f"using vsi={VSI_COL!r}, id={ID_COL!r}, signal={SIG_COL!r}")

    MIN_VSI = 0.7            # threshold from the paper; check it, do not trust it
    MIN_SIGNAL = 1.5         # ignore near-empty pixels

    px_df = cell_px
    if SIG_COL is not None:
        px_df = px_df.filter(pl.col(SIG_COL) > MIN_SIGNAL)

    per_cell = (
        px_df.group_by(ID_COL)
        .agg([
            pl.col(VSI_COL).mean().alias("vsi"),
            (pl.col(VSI_COL) < MIN_VSI).mean().alias("frac_low_vsi"),
            pl.len().alias("n_px"),
        ])
        .to_pandas()
    )
    per_cell[ID_COL] = per_cell[ID_COL].astype(str)
    per_cell = per_cell.set_index(ID_COL)
    print(f"VSI computed for {len(per_cell):,} cells")

In [ ]:
if HAVE_OVRLPY:
    # Do the ids actually match? Xenium cell_id in the transcript table and in
    # the count matrix should agree, but a mismatch here would silently give
    # every cell NaN — so check before trusting the join.
    overlap = per_cell.index.intersection(adata.obs_names)
    print(f"{len(overlap):,} ids match between ovrlpy output and adata.obs_names")
    if len(overlap) == 0:
        print("\nNO OVERLAP. Compare the two formats:")
        print("  ovrlpy :", list(per_cell.index[:3]))
        print("  adata  :", list(adata.obs_names[:3]))
        print("If one side has a suffix such as '-1', strip it before joining.")
    else:
        adata.obs["vsi"] = per_cell["vsi"].reindex(adata.obs_names)
        adata.obs["frac_low_vsi"] = per_cell["frac_low_vsi"].reindex(adata.obs_names)
        covered = adata.obs["vsi"].notna()
        print(f"{covered.sum():,} of {adata.n_obs:,} cells fall inside the imaged window")

In [ ]:
if HAVE_OVRLPY and "vsi" in adata.obs and adata.obs["vsi"].notna().any():
    sub = adata[adata.obs["vsi"].notna().to_numpy()]
    xs, ys = sub.obsm["spatial"].T

    fig, axes = plt.subplots(1, 3, figsize=(16, 4.8))

    axes[0].hist(sub.obs["vsi"], bins=50, color=NAVY)
    axes[0].axvline(MIN_VSI, color=GOLD, lw=2, label=f"VSI = {MIN_VSI}")
    axes[0].set_xlabel("mean VSI per cell"); axes[0].set_ylabel("cells")
    axes[0].legend(); sns.despine(ax=axes[0])

    pc = axes[1].scatter(xs, ys, c=sub.obs["vsi"], s=6, cmap="magma",
                         vmin=0.4, vmax=1.0, linewidths=0, rasterized=True)
    plt.colorbar(pc, ax=axes[1], label="mean VSI")
    axes[1].set_title("where is integrity low?")

    low = (sub.obs["vsi"] < MIN_VSI).to_numpy()
    axes[2].scatter(xs, ys, s=4, c="0.88", linewidths=0, rasterized=True)
    axes[2].scatter(xs[low], ys[low], s=8, c=CORAL, linewidths=0, rasterized=True)
    axes[2].set_title(f"cells below VSI {MIN_VSI}  (n={low.sum():,})")

    for ax in axes[1:]:
        ax.set_aspect("equal"); ax.invert_yaxis(); ax.set_xticks([]); ax.set_yticks([])
    plt.tight_layout(); plt.show()

    # do low-VSI cells favour particular cell types?
    tab = pd.crosstab(sub.obs["cell_type"], sub.obs["vsi"] < MIN_VSI, normalize="index")
    if True in tab.columns:
        print("\nfraction of each cell type flagged as low VSI:")
        print((100 * tab[True]).round(1).sort_values(ascending=False).to_string())

### What to conclude

Three things worth taking away, in increasing order of importance.

1. **Some of your "cells" are two cells.** Not because the polygon was drawn badly,
   but because the tissue is three-dimensional and your analysis is not.
2. **The flagged cells are not randomly distributed.** If one cell type is
   over-represented among low-VSI cells, that type is disproportionately affected by
   stacking — densely packed small cells (lymphocytes) and thin, spread-out cells
   (endothelium, fibroblasts) usually top the list. Any co-localisation result
   involving them inherits the problem.
3. **This is an independent check.** VSI never sees your segmentation, your
   clustering, or your annotation. When it disagrees with them, that is information —
   which is exactly the property that makes the segmentation-free analysis in the
   next section worth doing too.

The honest use of this in a paper is as a reported QC metric plus a sensitivity
analysis: show your main finding holds after excluding low-VSI cells. Deleting them
silently is not better than keeping them silently.

### Exercise 5.3
Re-run one result from notebook 04 — the neighbourhood enrichment heatmap is the
quickest — after dropping cells with `vsi < 0.7`. Does any pair of cell types change
its z-score materially? A co-localisation that disappears when you remove vertical
doublets was never a co-localisation; it was two cells being counted as one.

In [ ]:
# your code here

## 5. Before anyone drew a polygon

Every result so far rests on the segmentation. Here is how to check whether a
finding survives without it: rasterise the transcripts onto a grid and analyse the
grid. No cells, no polygons, no assumptions.

In [ ]:
BIN = 10  # um
gx = ((tx["x_location"] - tx["x_location"].min()) // BIN).astype(int)
gy = ((tx["y_location"] - tx["y_location"].min()) // BIN).astype(int)
nx, ny = gx.max() + 1, gy.max() + 1

def raster(gene=None):
    m = np.ones(len(tx), bool) if gene is None else (tx["feature_name"] == gene).to_numpy()
    img = np.zeros((ny, nx))
    np.add.at(img, (gy[m], gx[m]), 1)
    return img

total = raster()
picks = [g for g in ["EPCAM", "COL1A1", "PTPRC"] if g in set(tx["feature_name"])][:3]

fig, axes = plt.subplots(1, len(picks) + 1, figsize=(4.4 * (len(picks) + 1), 4.4))
axes[0].imshow(total, cmap="magma", vmax=np.percentile(total, 99))
axes[0].set_title(f"all transcripts, {BIN} um bins")
for ax, g in zip(axes[1:], picks):
    im = raster(g)
    ax.imshow(im, cmap="magma", vmax=max(np.percentile(im, 99.5), 1))
    ax.set_title(g)
for ax in axes:
    ax.axis("off")
plt.tight_layout(); plt.show()

> **Try it yourself — how coarse is too coarse?**
>
> `BIN` sets the grid square size in microns. Small bins are noisy; large bins blur
> structure away. A typical cell is 10–20 µm across — what happens when your bin is
> smaller than a cell, and when it is much bigger?

In [ ]:
TRY_BIN = 10        # <-- CHANGE THIS (try 3, 10, 25, 50)

gx2 = ((tx["x_location"] - tx["x_location"].min()) // TRY_BIN).astype(int)
gy2 = ((tx["y_location"] - tx["y_location"].min()) // TRY_BIN).astype(int)
img2 = np.zeros((gy2.max() + 1, gx2.max() + 1))
np.add.at(img2, (gy2, gx2), 1)

fig, ax = plt.subplots(figsize=(5.5, 5.5))
ax.imshow(img2, cmap="magma", vmax=np.percentile(img2, 99))
ax.axis("off"); ax.set_title(f"{TRY_BIN} um bins")
plt.show()

print(f"grid is {img2.shape[1]} x {img2.shape[0]} squares")
print(f"median transcripts per square: {np.median(img2[img2 > 0]):.0f}")

Compare these images with your cell-type map. **If a spatial pattern is visible in
the raw transcript density but absent from the cell-type map, your segmentation ate
it** — often the case for cells with little cytoplasm, or for regions where the
boundary stain failed.

This is the cheapest sanity check in the whole field and almost nobody does it.

### Exercise 5.4
Compute the correlation between two genes on the 10 um grid, and again between the
same two genes across cells. Do they agree? A pair that correlates on the grid but
not across cells is a candidate for a segmentation-driven artefact — or for a
genuine paracrine relationship between adjacent cells, which is exactly the
ambiguity you have to think through.

In [ ]:
# your code here

## 6. Where this leaves you

| Question | scRNA-seq | Xenium |
|---|---|---|
| Which cell types are present? | **better** — whole transcriptome, deeper | limited to the panel |
| Rare cell state discovery | **better** | panel has to already contain the markers |
| Which cells touch which? | impossible | direct |
| Expression vs distance to a structure | impossible | direct |
| Sub-cellular localisation | impossible | direct |
| Detecting vertical doublets | impossible | direct (ovrlpy) |
| Tissue compartments / niches | impossible | direct |
| Dissociation-sensitive cell types | badly biased | unbiased |
| Sample size | many cells, cheap | expensive per section, **n is the number of sections** |

The honest summary: **they answer different questions and the best studies use both.**
Discovery in dissociated data, placement and validation in situ. Anyone telling you
one replaces the other is selling something.

---
**Next:** `06_design_your_own.ipynb`